# Working with solvers

Linopy hands the model off to a solver backend &mdash; HiGHS, Gurobi, CPLEX, CBC, GLPK, SCIP, Xpress, MOSEK, MindOpt, COPT, Knitro, or the GPU solver cuPDLPx. This notebook walks through:

- the standard ``model.solve(...)`` workflow,
- inspecting the solver afterwards via ``model.solver`` and ``SolverReport``,
- the lower-level *construct-then-solve* API for advanced use,
- querying solver capabilities through ``SolverFeature``,
- listing installed and licensed solvers.

## A small example model

Minimize $x + 2y$ subject to $x, y \ge 0$, $3x + 7y \ge 10$, $5x + 2y \ge 3$.

In [ ]:
from linopy import Model


def build_model():
    m = Model()
    x = m.add_variables(lower=0, name="x")
    y = m.add_variables(lower=0, name="y")
    m.add_constraints(3 * x + 7 * y >= 10)
    m.add_constraints(5 * x + 2 * y >= 3)
    m.add_objective(x + 2 * y)
    return m


m = build_model()
m

## The recommended path: ``model.solve(...)``

``model.solve`` picks an installed solver (first entry of ``linopy.available_solvers`` unless ``solver_name`` is given), runs it, writes the solution back into the variables, and returns a ``(status, termination_condition)`` tuple.

In [ ]:
status, termination = m.solve(solver_name="highs")
status, termination

In [ ]:
f"Objective={m.objective.value}", m.solution.to_pandas()

## Inspecting the solver after solving

After ``model.solve(...)`` the solver instance stays attached to the model as ``model.solver``. You can read off the solver name, the native solver model, the status and &mdash; new in this release &mdash; a ``SolverReport`` with runtime, MIP gap, dual (best) bound, and iteration counts.

In [ ]:
m.solver

In [ ]:
m.solver_name, type(m.solver_model).__name__

In [ ]:
m.solver.report

Not every backend fills in every field of ``SolverReport`` &mdash; if a solver doesn't expose a value it stays ``None``. ``mip_gap`` and ``dual_bound`` are most informative on MIPs.

Some solvers (Gurobi, MOSEK, &hellip;) hold a license while the underlying handle is alive. You can release it explicitly:

In [ ]:
m.solver.close()  # frees the native handle (and license)
# or, to also detach the wrapper:
m.solver = None
m.solver, m.solver_name

## Advanced: construct-then-solve

For most users ``model.solve(...)`` is enough. If you want more control &mdash; e.g. to poke at the native solver model before running it, or to obtain the ``Result`` object directly &mdash; you can build the solver in one step and run it in another:

In [ ]:
from linopy.solvers import Solver

m = build_model()
solver = Solver.from_name("highs", m, io_api="direct")
solver

``solver.solver_model`` is the native solver handle &mdash; here a ``highspy.Highs`` instance. You could tweak it directly before running:

In [ ]:
type(solver.solver_model).__name__

In [ ]:
result = solver.solve()
result

``Result`` carries the status, solution, solver name, and report. Writing it back into the `Model`, combining numeric values with labels and coordinates, is a separate call:

In [ ]:
m.assign_result(result)
m.solution

Equivalently, each solver subclass has a ``from_model`` classmethod:

```python
from linopy.solvers import Highs
solver = Highs.from_model(m, io_api="direct")
```

## Querying solver capabilities

Each ``Solver`` subclass declares its capabilities as a set of ``SolverFeature`` values. Use ``SolverClass.supports(feature)`` to check a specific one, or ``SolverClass.supported_features()`` to list all of them.

In [ ]:
from linopy import SolverFeature
from linopy.solvers import Highs

(
    Highs.supports(SolverFeature.QUADRATIC_OBJECTIVE),
    Highs.supports(SolverFeature.IIS_COMPUTATION),
)

In [ ]:
import pandas as pd

pd.Series(
    {f.name: Highs.supports(f) for f in SolverFeature},
    name="Highs",
)

This is useful to gate solver-specific code paths, for example checking ``GPU_ACCELERATION`` before dispatching to cuPDLPx, or ``IIS_COMPUTATION`` before calling ``model.compute_infeasibilities()``.

## Which solvers do I have?

Two registries are exposed at the top level:

- ``linopy.available_solvers`` &mdash; solvers whose Python package or binary is **installed**. Cheap; does not acquire a license.
- ``linopy.licensed_solvers`` &mdash; the subset that currently passes a **license** probe. Useful in tests or to pick a solver at runtime.

In [ ]:
import linopy

list(linopy.available_solvers)

In [ ]:
list(linopy.licensed_solvers)

Both are lazy and refreshable &mdash; call ``linopy.available_solvers.refresh()`` after installing or licensing a new solver in the same process. For a per-solver probe use ``SolverClass.license_status()``, which returns a ``LicenseStatus`` dataclass:

In [ ]:
Highs.license_status()

In [ ]:
from linopy.solvers import check_solver_licenses

check_solver_licenses("highs")

## Summary

- ``model.solve(...)`` is the recommended entry point and leaves a fully-formed ``Solver`` on ``model.solver`` afterwards.
- ``model.solver.report`` gives runtime, MIP gap, dual bound, and iterations on supporting solvers.
- ``Solver.from_name(...).solve()`` plus ``model.assign_result(result)`` is the explicit two-step path for advanced cases.
- ``SolverFeature`` and ``SolverClass.supports(...)`` let you check what a backend can do without trying it.
- ``linopy.available_solvers`` is *installed*; ``linopy.licensed_solvers`` is *installed and licensed*.